# Day 3 — Solution: R², Standard Errors, t-Stats

In [ ]:
import os, sys, pathlib
root = pathlib.Path.cwd()
for _ in range(6):
    if (root / "qrc").is_dir():
        break
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import numpy as np
import pandas as pd
from scipy import stats
DATA_SOURCE = os.environ.get("QRC_DATA", "real")
from qrc.data import get_prices
from qrc.synth import synthetic_prices

## E1 — `ols_fit()`, verified against statsmodels

In [ ]:
def ols_fit(x, y):
    x = np.asarray(x, float); y = np.asarray(y, float)
    n = len(x)
    X = np.column_stack([np.ones(n), x])
    XtX_inv = np.linalg.inv(X.T @ X)          # small k: inv is fine HERE; formulas, not production solves
    b = XtX_inv @ X.T @ y
    e = y - X @ b
    s2 = e @ e / (n - 2)
    vcov = s2 * XtX_inv
    se = np.sqrt(np.diag(vcov))
    r2 = 1 - (e @ e) / ((y - y.mean()) @ (y - y.mean()))
    return {"alpha": b[0], "beta": b[1], "se_alpha": se[0], "se_beta": se[1],
            "t_alpha": b[0]/se[0], "t_beta": b[1]/se[1], "r2": r2, "resid": e}

rng = np.random.default_rng(3)
m = rng.normal(0.0005, 0.011, 1200)
y = 0.0002 + 0.80 * m + rng.normal(0, 0.010, 1200)
f = ols_fit(m, y)
print({k: round(v, 5) for k, v in f.items() if k != "resid"})

import statsmodels.api as sm
smf = sm.OLS(y, sm.add_constant(m)).fit()
checks = {"params": np.abs(np.asarray(smf.params) - [f["alpha"], f["beta"]]).max(),
          "bse":    np.abs(np.asarray(smf.bse) - [f["se_alpha"], f["se_beta"]]).max(),
          "t":      np.abs(np.asarray(smf.tvalues) - [f["t_alpha"], f["t_beta"]]).max(),
          "r2":     abs(smf.rsquared - f["r2"])}
print({k: f"{v:.2e}" for k, v in checks.items()})

**Expected reasoning.** Every quantity matches at 1e-12 or better — your
matrix code *is* what the library computes. Look at the fit itself: β̂ ≈
0.78–0.82 with SE ≈ 0.010/(0.011·√1200) ≈ 0.026 — so t(β̂) ≈ 31: the
slope is nailed (the world plants it loudly), while α̂'s t hovers ≈ 1.8:
**within the same regression, one coefficient can be certain and the other
unknown.** R² ≈ 0.44 here — σ_e vs σ_m arithmetic
(β²σ_m²/(β²σ_m²+σ_e²) ≈ 7.7e-5/(7.7e-5+10e-5) ≈ 0.44), not "model
quality". The takeaway
is procedural: *verify machinery on planted worlds; spend trust on real
ones.*

## E2 — CI coverage, 200 parallel worlds

In [ ]:
crit = stats.t.ppf(0.975, 1200 - 2)
covers_a, covers_b = 0, 0
for i in range(200):
    rng_i = np.random.default_rng(100 + i)
    yi = 0.0002 + 0.80 * m + rng_i.normal(0, 0.010, 1200)   # same X, fresh noise
    fi = ols_fit(m, yi)
    covers_a += abs(fi["alpha"] - 0.0002) <= crit * fi["se_alpha"]
    covers_b += abs(fi["beta"] - 0.80)  <= crit * fi["se_beta"]
print(f"95% CI coverage: alpha {covers_a/200:.2f}   beta {covers_b/200:.2f}")

**Expected pattern.** Both ≈ 0.92–0.98 (binomial SE at 200 runs ≈ 1.5% —
so even 0.91 is ordinary luck, report it as such). **The meaning, fixed
forever: a 95% CI is a procedure that traps the truth in ~95% of parallel
worlds — it is NOT "95% probability the truth lies here" for the one world
you got.** Frequentist discipline from module 04, now with regression
parameters: the interval is random; the truth is fixed.

## E3 — sampling scatter, seen

In [ ]:
def sim_betas(n, n_sims=500, seed0=1000):
    out = np.empty(n_sims)
    for i in range(n_sims):
        rg = np.random.default_rng(seed0 + i)
        mi = rg.normal(0.0005, 0.011, n)
        yi = 0.0002 + 0.80 * mi + rg.normal(0, 0.010, n)
        out[i] = ols_fit(mi, yi)["beta"]
    return out

for n in (756, 189):
    bs = sim_betas(n)
    se_formula = 0.010 / (0.011 * np.sqrt(n))
    print(f"n={n:4d}: sd(beta_hats) {bs.std(ddof=1):.4f} | SE formula {se_formula:.4f} | 95% range ({np.percentile(bs, 2.5):.3f}, {np.percentile(bs, 97.5):.3f})")

**Expected numbers.** sd of the 500 β̂'s ≈ the formula SE to 2–3 decimals;
n 756→189 multiplies scatter by √(756/189) = 2.0 — observed ratio within a
few % of 2.00. **β̂ is a draw from N(β, SE²) in these worlds; the SE
formula *is* the SD of repeats; "95% range" ≈ β ± 2SE (0.73–0.87 at three
years!).** Sit with that: three years of daily data and your beta is known
to ±0.07 — quote hedging ratios accordingly.

## E4 — the R² calibration ladder

In [ ]:
if DATA_SOURCE == "real":
    px = get_prices(["SPY", "QQQ", "AAPL"], start="2010-01-01")
    rr = np.log(px).diff().dropna()
    pair_idx, pair_stk = ("SPY", "QQQ"), ("SPY", "AAPL")
else:
    p1 = synthetic_prices(n_days=3800, n_assets=2, seed=5, corr=0.85)
    p2 = synthetic_prices(n_days=3800, n_assets=2, seed=23, corr=0.5)
    rr = pd.concat({"IDX1": np.log(p1["S0"]).diff(), "IDX2": np.log(p1["S1"]).diff(),
                    "MKT": np.log(p2["S0"]).diff(), "STK": np.log(p2["S1"]).diff()}, axis=1).dropna()
    rr.columns = ["SPY", "QQQ", "SPY_dup", "AAPL"]
    pair_idx, pair_stk = ("SPY", "QQQ"), ("SPY_dup", "AAPL")

for name, (xnam, ynam) in {"two indices": pair_idx, "stock ~ market": pair_stk}.items():
    f = ols_fit(rr[xnam].values, rr[ynam].values)
    print(f"{name:16s}: R2 {f['r2']:.3f} (corr^2 check {np.corrcoef(rr[xnam], rr[ynam])[0,1]**2:.3f})")
xm, ym = rr[pair_stk[0]].values, rr[pair_stk[1]].values
f_pred = ols_fit(xm[:-1], ym[1:])        # today's market -> tomorrow's stock
print(f"{'predictive (bad)':16s}: R2 {f_pred['r2']:.4f}  t(beta) {f_pred['t_beta']:+.2f}")

**Expected pattern.** Two indices: R² ≈ 0.7–0.9; stock ~ market: ≈ corr² ≈
0.2–0.4; predictive-on-returns: ≈ 0.00 with |t| ≲ 2 (if it is not
≈ 0 in **your** run, look for the bug before celebrating — autocorrelated
x, an unshifted y, or overlapping windows are the usual suspects; module
08 owns this topic). The calibration bands from the lesson are honored:
**high R² belongs to near-duplicate instruments; stock-on-market is a
minority-explained regression; honest predictability of raw returns is
vanishingly small in-sample too, most days.**

## E5 — the quote repair (exemplar)

Repaired: "On 2520 daily observations (2010–2019) the factor loading was
1.35 with a standard error of 0.12 — the estimate exceeds 1 by ~2.9 SEs,
consistent with amplified exposure *in that window*; the regression
explains 87% of the return variance, as expected between two
broad-market-linked series." The caveats: (i) loading ≠ loading-tomorrow
(window dependence); (ii) high R² between related instruments is
mechanical, not validating.

Instant distrust triggers for the 0.87: the regression was run on **levels
or overlapping sums** (day 8), the series are near-identical instruments
(QQQ vs a Nasdaq tracker — R² measures redundancy), or residuals are
autocorrelated, in which case R² overstates explanatory content and the
SEs in the slide are fiction.

**Common mistakes.** Trusting a CI computed from SEs you haven't verified
against the data's behavior; comparing R² across different questions ("our
model beats theirs, 0.9 vs 0.3") — R² is within-question only; reading
t(α̂) ≈ 1 as "α = 0, done" rather than "α is imprecisely measured —
report the CI and move on."